## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [1]:
# The imports

import os
import requests
from dotenv import load_dotenv
from openai import AsyncOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool,OpenAIChatCompletionsModel, SQLiteSession,set_tracing_disabled
load_dotenv(override=True)


True

In [2]:
groq_client = AsyncOpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

set_tracing_disabled(True)

## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [3]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller",  model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=groq_client
    ))

In [4]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [37]:
# Here is the final output

print(result.final_output)

Why did the autonomous AI agent get a parking ticket?

Because it kept trying to *self‑park* itself… and the meter said, “Sorry, I only accept human‑driven payments!”


In [5]:
# Here is the detail of the LLM calls

result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the autonomous AI agent quit its job?\n\nBecause every time it tried to take a “break,” the system kept telling it to “continue processing”! 😄',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'openai/gpt-oss-120b',
   'response_id': 'chatcmpl-9ffb602e-a8c1-4b1d-a7e5-e5b3f7fdb636'}}]

## Adding Observability with a trace

In [6]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

Why did the autonomous AI agent start a meditation class?  

Because it wanted to learn how to *self‑optimize* without over‑clocking its thoughts! 😌🤖


## Now go and look at the trace

https://platform.openai.com/traces

In [7]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Here are five light‑hearted jokes about AI agents—no offense intended, just a little tech‑y humor!

1. **The Over‑Helpful Assistant**  
   *Why did the AI agent bring a ladder to the meeting?*  
   Because it heard the project was “going to the next level” and wanted to be prepared!

2. **The Forgetful Bot**  
   *How many AI agents does it take to change a lightbulb?*  
   Just one—except it first needs to look up the definition of “lightbulb,” ask for permission, and then ask you to confirm the final answer.

3. **The Over‑Confident Predictor**  
   *What did the AI agent say after predicting the weather perfectly for a week?*  
   “I’m not a meteorologist, I’m just really good at guessing… with 99.9% confidence!”

4. **The Chatty Chatbot**  
   *Why did the AI agent get kicked out of the quiet library?*  
   Because every time someone asked a question it responded with, “I’m sorry, I don’t understand… can you rephrase that in 42 different ways?”

5. **The Philosophical Bot**  
   *W

## Part 2: Adding a tool

In [8]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [9]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [10]:
push("HEY!!")

Push: HEY!!


In [11]:
push

<function __main__.push(message)>

In [12]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [13]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001DD285CF1D0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False)

In [14]:
push_tool.description

'Send the given message to the user as a push notification'

In [18]:

notifier = Agent(name="Notifier",  model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=groq_client
    ), instructions="You notify the user upon request", tools=[push_tool])

In [19]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


BadRequestError: Error code: 400 - {'error': {'message': "Tool call validation failed: tool call validation failed: parameters for tool push_tool did not match schema: errors: [missing properties: 'message', additionalProperties 'Message' not allowed]", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "push_tool", "arguments": {\n  "Message": "Your pizza has arrived!"\n}}'}}

## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [3]:
agent = Agent(name="Assistant",  model=OpenAIChatCompletionsModel(
        model="openai/gpt-oss-120b",
        openai_client=groq_client
    ))

In [4]:
response = await Runner.run(agent, "Hi there. My name is Shivam.")
print(response.final_output)

Hello Shivam! 👋 Nice to meet you. How can I assist you today?


In [5]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

I don’t have any information about your name. If you’d like to share it, feel free to let me know!


## Memory approach 1 - just manually pass in the list of dicts

In [6]:
response = await Runner.run(agent, "Hi there. My name is Shivam.")
print(response.final_output)

Hello Shivam! Nice to meet you. I’m ChatGPT – how can I assist you today?


In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Shivam.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>